# cordslite API

In [ ]:
#| default_exp core

In [ ]:
#| export
from datetime import datetime, timedelta, timezone
from fasthtml.common import *
from fastcore.meta import *
from fastcore.utils import *
from nacl.bindings import crypto_aead_xchacha20poly1305_ietf_decrypt as xchacha_decrypt

import asyncio,ffmpeg,httpx,json,opuslib_next,os,random,re,socket,time
import websockets.asyncio.client

In [ ]:
from IPython.display import Audio

Discord has three main APIs:
- **REST API** - for actions (send message, get channels, fetch history). Request/response style.
- **Gateway API** - for real-time events via WebSocket (new messages, reactions, user joins).
- **Voice API** - for real-time streaming of audio via UDP

We start with the REST API. To use it, you need a **bot token** from the [Discord Developer Portal](https://discord.com/developers/applications):
1. Create an application
2. Go to "Bot" section and create a bot
3. Copy the token (keep it secret!)
4. Under "OAuth2 → URL Generator", select `bot` scope and choose permissions (e.g., `Send Messages`, `Read Message History`)
5. Use the generated URL to invite the bot to your server

The `DiscordClient` wraps `httpx.Client` with Discord's base URL (`https://discord.com/api/v10`) and auth headers pre-configured.

In [ ]:
#| export
class DiscordClient:
    def __init__(self, token=None, user_token=None, name='cordslite', ver='0.1'):
        self.token = token or os.environ['DISCORD_BOT_TOKEN']
        self.user_token = user_token or os.environ.get('DISCORD_USER_TOKEN')
        self.base_url = 'https://discord.com/api/v10'
        self.headers = {'Authorization': f'Bot {self.token}', 'User-Agent': f'DiscordBot ({name}, {ver})'}
        self.cli = httpx.AsyncClient(base_url=self.base_url, headers=self.headers)


The token defaults to the `DISCORD_BOT_TOKEN` environment variable—no hardcoding secrets!

In [ ]:
dc = DiscordClient()


## REST endpoints

Every REST call goes through `_req`, which handles two concerns automatically:
- **Rate limiting** — if Discord returns `429 Too Many Requests`, we wait the `Retry-After` duration and retry.
- **Error handling** — non-2xx responses are raised as `DiscordError` with Discord's error code, message, and HTTP status, so callers don't need to check responses manually.

In [ ]:
#| export
class DiscordError(Exception):
    def __init__(self, code, msg, status): self.code,self.msg,self.status = code,msg,status
    def __str__(self): return f"DiscordError({self.status}): [{self.code}] {self.msg}"

@patch
async def _req(self:DiscordClient, method, path, data=None, files=None, use_user=False, **kw):
    kw = filter_values(kw, negate(is_(None)))
    if method=='GET': params,json = kw,None
    else: params,json = None,kw
    if use_user:
        if not self.user_token: raise ValueError("User token required for this operation")
        kw['headers'] = {'Authorization': self.user_token}
    r = await self.cli.request(method, path, data=data, files=files, params=params, json=json)
    if r.status_code == 429:
        await asyncio.sleep(float(r.headers.get('Retry-After', 1))+0.1)
        return await self._req(method, path, use_user=use_user, **kw)
    if r.status_code >= 400:
        try: d = r.json()
        except Exception: d = {}
        raise DiscordError(d.get('code', 0), d.get('message', r.text[:200]), r.status_code)
    if r.status_code == 204: return 'done'
    try: return r.json()
    except Exception as e: raise Exception(f"Failed to parse response: {r.text}; {e}")

Discord's REST endpoints return JSON with many fields. Rather than defining properties for every field, we use a flexible base class pattern:

`DiscordObject` provides `__getitem__` and `__getattr__` so you can access data as `obj.name` or `obj['name']`. The `__dir__` method enables autocomplete in notebooks and solveit dialogs—just type `obj.` and see all available fields!

This means when Discord adds new fields to their API, our code automatically supports them without changes.

In [ ]:
#| export
class DiscordObject(AttrDict):
    def __init__(self, data, client):
        super().__init__(data)
        self._client = getattr(client, '_client', client)
    
    async def __call__(self, meth, path, **kw): return await self._client._req(meth, path, **kw)
    def obj(self, nm, d): return globals()[nm](d, self)
    def coll(self, nm, ds): return globals()[nm+'s'](self.obj(nm,d) for d in ds)

Discord's hierarchy is **Guild → Channels → Messages**. A Guild (server) contains channels, and channels contain messages. Each has its own REST endpoints:
- `GET /guilds/{id}` - fetch guild info
- `GET /guilds/{id}/channels` - list channels
- `GET /channels/{id}/messages` - fetch messages
- `POST /channels/{id}/messages` - send a message

We build wrapper classes for each, inheriting from `DiscordObject` and just adding a nice `__repr__`.

In [ ]:
#| export
class Guild(DiscordObject):
    def __repr__(self): return f"Guild(id={self.id}, name={self.name!r})"

We use `@patch` from fastcore to add methods incrementally—great for interactive development where you want to test each piece as you build it. This `guild` method fetches guild data and wraps it in our `Guild` class.

In [ ]:
#| export
@patch
async def guild(self:DiscordClient, guild_id):
    return Guild(await self._req('GET', f'/guilds/{guild_id}'), self)

In [ ]:
# gid = '1493461895615873044'
gid = '1327046393453613076'
gld = await dc.guild(gid); gld

```python
Guild(id=1327046393453613076, name="natedog's server")
```

Channels have a `type` field: 0=text, 2=voice, 4=category. The `Channels` class inherits from `list` and adds `_repr_html_` for nice table display in notebooks/solveit. This pattern—a wrapper class plus a collection class with HTML repr—makes exploring the API much more pleasant.

In [ ]:
#| export
def html_table(items, hdrs, fn):
    if not items: return ""
    rows = (Tr(map(Td, fn(o))) for o in items)
    return to_xml(Table(Thead(Tr(map(Th, hdrs))), Tbody(rows), cls="prose"))

In [ ]:
#| export
class Channel(DiscordObject):
    @property
    async def guild(self):
        if not hasattr(self, '_gld'): self._gld = await self._client.guild(self.guild_id)
        return self._gld
    def __repr__(self): return f"Channel(id={self.id}, name={self.name!r}, type={self.type})"

class Channels(list):
    def _repr_html_(self): return html_table(self, ("ID", "Name", "Type"), lambda c: (c.id, c.name, c.type))

In [ ]:
#| export
@patch
async def channels(self:Guild, limit=None):
    data = await self('GET', f'/guilds/{self.id}/channels')
    if limit: data = data[:limit]
    return self.coll('Channel', data)

In [ ]:
chs = await gld.channels(5)
chs

ID,Name,Type
1327046393453613077,Text Channels,4
1327046393453613078,Voice Channels,4
1327046393453613079,general,0
1327046393453613080,General,2
1327954661960978512,private,0


In [ ]:
#| export
@patch
async def channel(self:DiscordClient, channel_id):
    return Channel(await self._req('GET', f'/channels/{channel_id}'), self)

In [ ]:
chid = '1327046393453613079' # '1493461896139903028'

In [ ]:
ch = await dc.channel(chid)
ch

```python
Channel(id=1327046393453613079, name='general', type=0)
```

Messages are the core of most bot functionality. Note that Discord returns messages in reverse chronological order (newest first), so we `reverse()` the data to get chronological order. The table shows a preview of content, author, and timestamp.

Note: To read message content, your bot needs the `MESSAGE_CONTENT` privileged intent enabled in the Developer Portal! Without it, the `content` field will be empty for messages not sent by your bot or mentioning it.

In [ ]:
#| export
class Message(DiscordObject):
    def __init__(self, data, dobj):
        super().__init__(data, dobj)
        self.raw_content = self.content
        mentions = {m['id']: m['username'] for m in data.get('mentions', [])}
        def repl(m): return f"@{mentions.get(m.group(1), 'unknown')}"
        self['content'] = re.sub(r'<@!?(\d+)>', repl, self.content)

    def __repr__(self): 
        author = self.author['username']
        preview = self.content[:30] + '…' if len(self.content) > 30 else self.content
        return f"Message(id={self.id}, author={author!r}, content={preview!r})"

    @property
    async def channel(self):
        if not hasattr(self, '_ch'): self._ch = Channel(await self('GET', f'/channels/{self.channel_id}'), self._client)
        return self._ch

In [ ]:
#| export
class Messages(list):
    def _repr_html_(self):
        return html_table(self, ("ID", "Author", "Content", "Date"),
            lambda m: (m.id, m.author['username'], m.content[:50], m.timestamp[:10]))

In [ ]:
#| export
@patch
async def messages(self:Channel, limit=50):
    data = await self('GET', f'/channels/{self.id}/messages', limit=limit)
    return self.coll('Message', reversed(data))

In [ ]:
msgs = await ch.messages(5); msgs

ID,Author,Content,Date
1494381457928618014,DBuddy,I'm replying to myself 🤓,2026-04-16
1494381464177999893,DBuddy,Here is a file!,2026-04-16
1494381474240270511,DBuddy,Test our event listener! Otters are awesome 🦦,2026-04-16
1494381547586064515,DBuddy,"Houston, do we have a problem?",2026-04-16
1494381604842504372,DBuddy,Did we re-identify?,2026-04-16


Guild search uses snowflake IDs for date filtering (`min_id`/`max_id`), but we autoconvert `'YYYY-MM-DD'` strings for convenience for `before/after`.

In [ ]:
#| export
depoch = 1420070400000
def date2snowflake(date_str):
    "Convert 'YYYY-MM-DD' to a Discord snowflake ID"
    dt = datetime.fromisoformat(date_str).replace(tzinfo=timezone.utc)
    return str(int((dt.timestamp() * 1000 - depoch) * (2**22)))

@patch
async def search(self:Guild, content=None, author_id=None, channel_id=None, mentions=None,
                 has=None, before=None, after=None, pinned=None, sort_by=None, sort_order=None,
                 offset=None, limit=None, use_user=False):
    "Search guild messages. `before`/`after` accept 'YYYY-MM-DD' strings or snowflake IDs."
    if before and not str(before).isdigit(): before = date2snowflake(before)
    if after and not str(after).isdigit(): after = date2snowflake(after)
    r = await self('GET', f'/guilds/{self.id}/messages/search', use_user=use_user,
        content=content, author_id=author_id, channel_id=channel_id,
        mentions=mentions, has=has, min_id=after, max_id=before, pinned=pinned,
        sort_by=sort_by, sort_order=sort_order, offset=offset, limit=limit)
    return self.coll('Message', [m[0] for m in r['messages']])

In [ ]:
await gld.search(after='2026-02-16', limit=5)

ID,Author,Content,Date
1494381604842504372,DBuddy,Did we re-identify?,2026-04-16
1494381547586064515,DBuddy,"Houston, do we have a problem?",2026-04-16
1494381474240270511,DBuddy,Test our event listener! Otters are awesome 🦦,2026-04-16
1494381464177999893,DBuddy,Here is a file!,2026-04-16
1494381457928618014,DBuddy,I'm replying to myself 🤓,2026-04-16


Sometimes you need to search by name rather than snowflake ID. `find_member` searches the guild's members by username, nickname, or display name using Discord's member search endpoint, and returns the first match's user ID. This makes it easy to chain into `search`.

In [ ]:
#| export
@patch
async def find_member(self:Guild, name):
    "Search guild members by name/nick, return first match's user ID or None"
    data = await self( 'GET', f'/guilds/{self.id}/members/search', query=name, limit=1)
    return data[0]['user']['id'] if data else None

In [ ]:
uid = await gld.find_member('jeremyhoward')
assert uid
await gld.search(author_id=uid, limit=5)

The `members` property fetches all guild members and returns their display names, preferring server nickname over global display name over username. Note: this requires the **Server Members Intent** enabled in the Developer Portal (Bot → Privileged Gateway Intents).

In [ ]:
#| export
@patch(as_prop=True)
async def members(self:Guild):
    "List all guild members' display names (nick > global_name > username)"
    data = await self('GET', f'/guilds/{self.id}/members', limit=1000)
    return [m.get('nick') or m['user'].get('global_name') or m['user']['username'] for m in data]

In [ ]:
(await gld.members)[:5]

['nathan', 'SearchBuddy', 'DBuddy', 'Dizcord Util Bot', 'Search Agent']

Sending messages is a POST request with JSON body. Pass `reply_id` to thread a reply under an existing message. For file attachments, we switch to `multipart/form-data`—Discord expects a `payload_json` field with the message JSON, plus `files[n]` fields for each file.

In [ ]:
#| export
@patch
async def send(self:Channel, content='', files=None, reply_id=None):
    "Send a message with optional file attachments"
    path = f'/channels/{self.id}/messages'
    mref = {'message_id': reply_id} if reply_id else None
    payload = filter_values(dict(content=content, message_reference=mref), negate(is_(None)))
    fs = [('files[%d]' % i, (f.name, f.read_bytes(), None)) for i,f in enumerate(map(Path, listify(files)))]
    r = await self('POST', path, data={'payload_json': json.dumps(payload)}, files=fs)
    return Message(r, self)

In [ ]:
msg = await ch.send('Hi, from Solveit!'); msg

```python
Message(id=1494383523015164087, author='DBuddy', content='Hi, from Solveit!')
```

In [ ]:
reply_msg = await ch.send("I'm replying to myself 🤓", reply_id=msg.id); reply_msg

```python
Message(id=1494383523946303661, author='DBuddy', content="I'm replying to myself 🤓")
```

In [ ]:
await msg.channel

```python
Channel(id=1327046393453613079, name='general', type=0)
```

In [ ]:
msg = await ch.send('Here is a file!', files=['../README.md']); msg

```python
Message(id=1494383525539872970, author='DBuddy', content='Here is a file!')
```

In [ ]:
#| export
@patch
@delegates(Guild.search, but=['channel_id'])
async def search(self:Channel, **kwargs): return await (await self.guild).search(channel_id=self.id, **kwargs)

In [ ]:
await ch.search(has='file', limit=1)

ID,Author,Content,Date
1494381464177999893,DBuddy,Here is a file!,2026-04-16


Discord messages can have file attachments. The `Attachment` class wraps them as `DiscordObject`s—giving you attribute access to `filename`, `size`, `content_type`, `url`, etc. The `fetch` method downloads the file content using the existing httpx client.

In [ ]:
#| export
class Attachment(DiscordObject):
    async def fetch(self): return (await self._client.cli.get(self.url)).content
    def __repr__(self): return f"Attachment(filename={self.filename!r}, size={self.size}, type={self.content_type})"
    def _repr_html_(self):
        if self.content_type.startswith('image/'):
            return f'<img src="{self.url}" style="max-width:400px"><br><small>{self.filename} ({self.size:,}B)</small>'
        return f'<code>{self.filename}</code> ({self.content_type}, {self.size:,}B)'

@patch(as_prop=True)
def attachments(self:Message): return [Attachment(a, self._client) for a in self.get('attachments', [])]

In [ ]:
atts = msg.attachments; atts

[Attachment(filename='README.md', size=28163, type=text/markdown; charset=utf-8)]

In [ ]:
readme = (await atts[0].fetch()).decode()
print(readme[:16])

# cordslite 🍺





DMs (Direct Messages) are just regular channels in Discord's API — no special handling needed! To start a DM conversation, POST to `/users/@me/channels` with a `recipient_id`. Discord returns a standard channel object, so `send()` and `messages()` work exactly as they do for guild channels.

To detect DMs in the gateway, check for the absence of `guild_id` — DM messages don't belong to any guild, so this field is missing or `None`. This makes it easy to route DM vs guild messages in your bot's handler.

In [ ]:
#| export
@patch
async def create_dm(self:DiscordClient, user_id):
    r = await self._req('POST', '/users/@me/channels', recipient_id=user_id)
    return Channel(r, self)

In [ ]:
# # Commented out so we don't spam Nate
# dm = await dc.create_dm('346450717025894400')  # nathan's user ID
# await dm.send('Hello from DMs!')

Members vs Users: A **User** is a global Discord account. A **Member** is a user *within a specific guild*—it has guild-specific data like nickname, roles, and join date. The `nick or user['username']` pattern shows the server nickname if set, otherwise falls back to the global username.

Note: The members endpoint requires the `GUILD_MEMBERS` privileged intent enabled in your bot settings on the Developer Portal!

In [ ]:
#| export
class User(DiscordObject):
    def __repr__(self): return f"User(id={self.id}, username={self.username!r})"

class Member(DiscordObject):
    def __repr__(self):
        name = self.nick or self.user['username']
        return f"Member(id={self.user['id']}, name={name!r}, joined={self.joined_at[:10]})"

class Members(list):
    def _repr_html_(self): return html_table(self, ("ID", "Name", "Joined", "Roles"), lambda m: (m.user['id'], m.nick or m.user['username'], m.joined_at[:10], len(m.roles)))

@patch
async def members(self:Guild, limit=100):
    data = await self('GET', f'/guilds/{self.id}/members', limit=limit)
    return self.coll('Member', data)

In [ ]:
mems = await gld.members(5); mems

ID,Name,Joined,Roles
346450717025894400,nathan,2025-01-09,0
1327047896436178954,SearchBuddy,2025-01-09,1
1361823507679543306,DBuddy,2025-04-15,1
1448038710229733398,Dizcord Util Bot,2025-12-09,1
1467222191182712986,Search Agent,2026-01-31,1


In [ ]:
#| export
@patch
async def search_all(self:Guild, limit=500, delay=1.0, max_age_days=None, show=False, **kwargs):
    "Paginated search returning up to `limit` messages"
    all_msgs, offset = [], 0
    while len(all_msgs) < limit:
        msgs = await self.search(offset=offset, limit=25, **kwargs)
        if not msgs: break
        if max_age_days is not None:
            cutoff = datetime.now(timezone.utc) - timedelta(days=max_age_days)
            msgs = [m for m in msgs if datetime.fromisoformat(m.timestamp) > cutoff]
        all_msgs.extend(msgs)
        offset += 25
        if show: print(f'Fetched {len(msgs)} more')
        if len(msgs) < 25: break
        await asyncio.sleep(delay)
    return Messages(all_msgs[:limit])

@patch
@delegates(Guild.search_all, but=['channel_id'])
async def search_all(self:Channel, **kwargs):
    gld = await self.guild
    return await gld.search_all(channel_id=self.id, **kwargs)

In [ ]:
# r = await ch.search_all()
# len(r)

In [ ]:
#| export
@patch
async def delete(self:Message):
    "Delete this message"
    return await self('DELETE', f'/channels/{self.channel_id}/messages/{self.id}')

@patch
async def delete_message(self:Channel, message_id):
    "Delete a message by ID"
    return await self('DELETE', f'/channels/{self.id}/messages/{message_id}')

@patch
async def bulk_delete(self:Channel, message_ids):
    "Bulk delete messages (must be <14 days old)"
    return await self('POST', f'/channels/{self.id}/messages/bulk-delete', messages=[str(i) for i in message_ids])

In [ ]:
#| export
@patch
async def create_thread(self:Message, name):
    "Create a thread from this message"
    r = await self('POST', f'/channels/{self.channel_id}/messages/{self.id}/threads', name=name)
    return Channel(r, self)

@patch
async def thread(self:DiscordClient, thread_id):
    "Fetch a thread (which is a Channel)"
    return Channel(await self._req('GET', f'/channels/{thread_id}'), self)

In [ ]:
#| export
@patch
async def _del_rotating(self:Channel, message_ids, delay=0.5, show=False):
    if show: print(f'Deleting {len(message_ids)} messages...')
    for i,mid in enumerate(message_ids):
        use_user = i % 2 == 1 and bool(self._client.user_token)
        await self('DELETE', f'/channels/{self.id}/messages/{mid}', use_user=use_user)
        await asyncio.sleep(delay)
        if show: print('.', end='', flush=True)
    return len(message_ids)

@patch
async def search_and_delete_all(self:Channel, content, delay=2, show=False, **kwargs):
    "Bulk delete recent msgs, individually delete older ones"
    assert content, "Must include content to search for"
    cutoff = datetime.now(timezone.utc) - timedelta(days=13)
    gld = await self.guild
    msgs = await gld.search_all(content=content, channel_id=self.id, show=show, **kwargs)
    recent = [m for m in msgs if datetime.fromisoformat(m.timestamp) > cutoff]
    old = [m for m in msgs if datetime.fromisoformat(m.timestamp) <= cutoff]
    for i in range(0, len(recent), 20):
        ms = recent[i:i+20]
        await self.bulk_delete([m.id for m in ms])
        if show: print(f'Bulk deleted {len(ms)}')
        await asyncio.sleep(delay)
    if old: await self._del_rotating([m.id for m in old], show=show, delay=delay)
    return len(msgs)

## Gateway API

The Gateway is Discord's WebSocket connection for **real-time events**. Unlike the REST API (request/response), the Gateway pushes events to you as they happen—new messages, reactions, user joins, etc.

The connection lifecycle is:
1. Fetch WSS URL from `/gateway/bot`
2. Connect to WebSocket
3. Receive **Hello** event with heartbeat interval
4. Start heartbeat loop to keep connection alive
5. Send **Identify** with your token and intents
6. Receive **Ready** event—you're connected!

**Intents** tell Discord which events you want. They're bitwise flags you OR together:
- `1 << 0` = GUILDS (guild/channel events)
- `1 << 9` = GUILD_MESSAGES (message events)
- `1 << 15` = MESSAGE_CONTENT (privileged—see message content)

In [ ]:
#| export
class GatewayClient:
    def __init__(self, intents, client, token=None):
        self.intents,self.dc = intents,client
        self.token = token or os.environ['DISCORD_BOT_TOKEN']
        self.ws = self.hb_int = self.session_id = self.seq = None
        self.running = False
        gw_info = httpx.get(f'{client.base_url}/gateway/bot', headers={'Authorization': f'Bot {self.token}'}).json()
        if 'url' not in gw_info: raise ConnectionError(f"Gateway auth failed: {gw_info.get('message', gw_info)}")
        self.url = f"{gw_info['url']}?v=10&encoding=json"
        self.handlers = {'READY': self.on_rdy}
    
    async def on_rdy(self, evt):
        self.session_id, self.user_id, = evt['session_id'], evt['user']['id']
        self.resume_url = evt.get('resume_gateway_url', self.url) + '?v=10&encoding=json'
        
    def __repr__(self): return f"GatewayClient({self.intents=}, {self.url=})"

In [ ]:
# For now, let's use basic intents for guilds and messages
intents = (1 << 0) | (1 << 9) | (1 << 15)  # GUILDS | GUILD_MESSAGES | MESSAGE_CONTENT

gc = GatewayClient(intents, dc); gc

GatewayClient(self.intents=33281, self.url='wss://gateway.discord.gg?v=10&encoding=json')

`Op` is a thin helper for constructing Gateway opcodes. Discord's Gateway expects JSON payloads with an `op` field (operation code) and `d` field (payload data). Rather than hand-building dicts each time, `Op.identify()` and `Op.heartbeat()` return properly structured payloads ready to send over the WebSocket.

In [ ]:
#| export
class Op(AttrDict):
    def __repr__(self): return f"Op(op={self.op}, d={self.d})"
    @classmethod
    def identify(cls, token, intents):
        d = AttrDict(token=token, intents=intents, properties=dict(os='linux',browser='discord_wrapper', device='discord_wrapper'))
        return cls(op=2, d=d)
    @classmethod
    def heartbeat(cls, seq): return cls(op=1, d=seq)
    @classmethod
    def resume(cls, token, session_id, seq): return cls(op=6, d=AttrDict(token=token, session_id=session_id, seq=seq))

The Gateway sends events as JSON with four fields:
- **`op`** — operation code: `0` = dispatch (real events), `1` = heartbeat request, `10` = hello, `11` = heartbeat ACK
- **`t`** — event type name (only for dispatches): `MESSAGE_CREATE`, `GUILD_CREATE`, etc.
- **`s`** — sequence number (only for dispatches): increments per event, needed for heartbeats and resuming
- **`d`** — the payload data

We track the latest `s` value so heartbeats can tell Discord "I've received everything up to here." The `Event` class wraps raw JSON and auto-converts known dispatch types (like `MESSAGE_CREATE`) into their corresponding wrapper classes (`Message`, `Channel`, etc.) via `evt_typs`.

In [ ]:
#| export
evt_typs = {'MESSAGE_CREATE': Message,
            'MESSAGE_UPDATE': Message,
            'MESSAGE_DELETE': Message,
            'GUILD_CREATE': Guild,
            'CHANNEL_CREATE': Channel}

class Event(DiscordObject):
    def __init__(self, data, client):
        super().__init__(data, client)
        typ = evt_typs.get(self.t)
        self.d = typ(self.d, client) if typ else self.d
    @property
    def type(self): return self.t
    @property
    def seq(self): return self.s
    def __repr__(self): return f"Event({self.op=}, {self.type=}, {self.seq=})"

In [ ]:
#| export
@patch
async def send(self:websockets.asyncio.client.ClientConnection, msg, **kw):
    if isinstance(msg, dict): msg = json.dumps(msg)
    return await self._orig_send(msg, **kw)

`recv_evt` is the low-level building block for receiving events. It reads one raw WebSocket message, wraps it as an `Event` (auto-converting known dispatch types), and updates the sequence counter. You can use it directly to pull events one at a time — useful for debugging or understanding what Discord is sending.

In [ ]:
#| export
@patch
async def recv_evt(self:GatewayClient):
    evt = Event(json.loads(await self.ws.recv()), self.dc)
    if evt.op == 0 and evt.seq: self.seq = evt.seq
    return evt

Rather than special-casing the READY event inside `_listen`, we register it as a regular dispatch handler via `gc.on()`. When Discord confirms a successful Identify, `on_rdy` caches `session_id`, `user_id`, and `resume_gateway_url` — the three values needed to Resume after a disconnect. Note the query params appended to `resume_gateway_url` — Discord returns it bare, but the docs require the same version and encoding as the initial connection.

In [ ]:
#| export
@patch
def on(self:GatewayClient, event_type, handler): self.handlers[event_type] = handler

The gateway runs on two cooperating loops. `_hb` sends heartbeats at the interval Discord specifies and watches for ACK responses — if one is missed, the connection is zombied and gets closed with code 4000 (which preserves the session for Resuming). `_listen` reads events and dispatches them by opcode. On disconnect (exception from `recv_evt`), it calls `_reconnect` to swap in a fresh WebSocket. The new connection triggers Hello (op 10), which restarts the heartbeat and sends either Identify (first connect) or Resume (reconnect) depending on whether a session already exists. Op 7 (Reconnect) follows the same path — close and reopen.

All reconnection funnels through `_reconnect`, which closes with code 4000 (keeping the session valid) and opens a new socket to `resume_gateway_url`. The listener's op 10 handler then takes over automatically.

In [ ]:
#| export
@patch
async def _hb(self:GatewayClient):
    await asyncio.sleep(self.hb_int / 1_000 * random.random()) # jitter
    while self.running:
        self._got_ack = False
        try: await self.ws.send(Op.heartbeat(self.seq))
        except Exception: return
        await asyncio.sleep(self.hb_int / 1_000)
        if not self._got_ack: return await self.ws.close(code=4000)

In [ ]:
#| export
@patch
async def _reconnect(self:GatewayClient, resume=False):
    code, url = 4000, self.resume_url
    if not resume:
        code, url = 1000, self.url
        self.session_id = None
    await self.ws.close(code=code)
    await asyncio.sleep(2)
    self.ws = await websockets.connect(url)

In [ ]:
#| export
@patch
async def _listen(self:GatewayClient):
    while self.running:
        try:
            evt = await self.recv_evt()
            if self.debug: print('DEBUG: Received Opcode:', evt.op)
            if evt.op == 10:
                self.hb_int = evt.d['heartbeat_interval']
                if hasattr(self, '_hb_task') and self._hb_task: self._hb_task.cancel()
                self._hb_task = asyncio.create_task(self._hb())
                if self.session_id: await self.ws.send(Op.resume(self.token, self.session_id, self.seq))
                else: await self.ws.send(Op.identify(self.token, self.intents))
            elif evt.op == 0 and evt.type in self.handlers: asyncio.create_task(self.handlers[evt.type](evt.d))
            elif evt.op == 11: self._got_ack = True
            elif evt.op == 1: await self.ws.send(Op.heartbeat(self.seq))
            elif evt.op == 7: await self._reconnect()
            elif evt.op == 9: await self._reconnect(resume=evt.d)
        except Exception as e:
            print(e)
            await self._reconnect()
            continue

`start` opens the initial WebSocket and spawns the listener — nothing else. The listener handles Hello, kicks off the heartbeat, and sends Identify, so the full connection lifecycle is driven by events rather than imperative setup.

In [ ]:
#| export
@patch
async def start(self:GatewayClient, debug=False):
    self.debug = debug
    self.running = True
    self.ws = await websockets.connect(self.url)
    self._listen_task = asyncio.create_task(self._listen())

In [ ]:
await gc.start(debug=True)

DEBUG: Received Opcode: 10


DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0


DEBUG: Received Opcode: 11


sent 4000 (private use); then received 4000 (private use)


DEBUG: Received Opcode: 10


...

sent 1000 (OK); then received 1000 (OK)


DEBUG: Received Opcode: 10


DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0
DEBUG: Received Opcode: 0


DEBUG: Received Opcode: 0
DBuddy: Did we re-identify?


DEBUG: Received Opcode: 11


Let's now test that our gateway is getting events. Here is a simple handlers for new messages.

In [ ]:
async def on_msg(msg):
    if not msg.author.get('bot'): return
    print(f"{msg.author['username']}: {msg.content}")

gc.on('MESSAGE_CREATE', on_msg)

Now, we can send a message and it should be printed out.

In [ ]:
await ch.send('Test our event listener! Otters are awesome 🦦')

```python
Message(id=1494383536193536111, author='DBuddy', content='Test our event listener! Otter…')
```

Let's trigger and reconnection that resumes and one that doesn't and see if everything works correctly and we still end up with our printed message.

In [ ]:
await gc._reconnect(resume=True)
await asyncio.sleep(2)
await ch.send('Houston, do we have a problem?')

```python
Message(id=1494383629688901683, author='DBuddy', content='Houston, do we have a problem?')
```

In [ ]:
await gc._reconnect(resume=False)
await asyncio.sleep(2)
await ch.send('Did we re-identify?')

```python
Message(id=1494383657929019646, author='DBuddy', content='Did we re-identify?')
```

`stop` is a full shutdown — cancels both background tasks and closes the WebSocket with the default code (1000), which tells Discord to invalidate the session. After this, a fresh `start()` would need to Identify from scratch.

In [ ]:
#| export
@patch
async def stop(self:GatewayClient):
    self.running = False
    for t in (self._hb_task, self._listen_task):
        if t: t.cancel()
    if self.ws: await self.ws.close()

In [ ]:
await gc.stop()

## Voice API

In [ ]:
await gc.start()
vch = chs[3]; vch

Connected! Session: 071532487214aa5877a8dcc277014457, heartbeat: 41250ms
Gateway started!


Channel(id=1327046393453613080, name='General', type=2)

Voice requires **three simultaneous connections**:

1. **Main Gateway WebSocket** (`gc`) — to request joining a voice channel and receive voice server info
2. **Voice Gateway WebSocket** — a *separate* WebSocket to a dedicated voice server for session coordination
3. **Voice UDP** — a UDP (datagram) connection for actual audio data. UDP is used over TCP because real-time audio needs low latency more than guaranteed delivery

`VoiceClient` manages the voice gateway WebSocket and UDP connections for a single voice channel, coordinated through the main `GatewayClient`.

In [ ]:
#| export
class VoiceClient:
    def __init__(self, gateway, channel):
        self.gw, self.ch = gateway, channel
        self.ws,self.udp,self.hb_int,self.ssrc,self.secret = None,None,None,None,None
        self.running,self.v_seq,self.loop = False,-1,None
    def __repr__(self): return f'VoiceClient({self.ch=}, {self.running=})'

In [ ]:
vc = VoiceClient(gc, vch); vc

VoiceClient(self.ch=Channel(id=1327046393453613080, name='General', type=2), self.running=False)

Voice requires additional `Op` opcodes beyond the main gateway's identify/heartbeat. These handle the voice-specific protocol: requesting to join a channel (`voice_state`), authenticating with the voice server (`voice_identify`), negotiating encryption (`select_protocol`), signaling audio intent (`speaking`), and keeping the voice connection alive (`voice_heartbeat`).

The voice heartbeat uses a different format from the main gateway — v8 requires `seq_ack` tracking the last received sequence number. It must start **immediately** after receiving Hello, before UDP setup, or the voice WebSocket will time out (close code 4006).

In [ ]:
#| export
@patch(cls_method=True)
def voice_state(cls:Op, guild_id, channel_id):
    return cls(op=4, d=AttrDict(guild_id=guild_id, channel_id=channel_id, self_mute=False, self_deaf=False))
@patch(cls_method=True)
def voice_identify(cls:Op, server_id, user_id, session_id, token):
    return cls(op=0, d=AttrDict(server_id=server_id, user_id=user_id, session_id=session_id, token=token))
@patch(cls_method=True)
def select_protocol(cls:Op, ip, port, mode='aead_xchacha20_poly1305_rtpsize'):
    return cls(op=1, d=AttrDict(protocol='udp', data=dict(address=ip, port=port, mode=mode)))
@patch(cls_method=True)
def speaking(cls:Op, ssrc, speaking=0):
    return cls(op=5, d=AttrDict(speaking=speaking, delay=0, ssrc=ssrc))
@patch(cls_method=True)
def voice_heartbeat(cls:Op, seq_ack=-1):
    return cls(op=3, d=AttrDict(t=int(time.time() * 1000), seq_ack=seq_ack))

`_connect` handles the voice WebSocket handshake: request to join via the main gateway, wait for Discord to assign a voice server, connect to it, authenticate, and start the voice heartbeat. The heartbeat must begin immediately after Hello — before UDP setup — or the voice WebSocket times out (close code 4006).

In [ ]:
#| export
@patch
async def _vhb(self:VoiceClient):
    await asyncio.sleep(self.hb_int / 1_000 * random.random())
    while self.running:
        await self.ws.send(Op.voice_heartbeat(self.v_seq))
        await asyncio.sleep(self.hb_int / 1_000)

@patch
async def _connect(self:VoiceClient):
    self.v_srv = {}
    srv_rdy = asyncio.Event()
    async def on_vs(data):
        self.v_srv.update(data)
        srv_rdy.set()
    self.gw.on('VOICE_SERVER_UPDATE', on_vs)
    await self.gw.ws.send(Op.voice_state(self.ch.guild_id, self.ch.id))
    await srv_rdy.wait()
    self.ws = await websockets.connect(f"wss://{self.v_srv['endpoint']}?v=8")
    hello = json.loads(await self.ws.recv())
    self.hb_int = hello['d']['heartbeat_interval']
    self.running = True
    asyncio.create_task(self._vhb())
    await self.ws.send(Op.voice_identify(self.ch.guild_id, self.gw.user_id, self.gw.session_id, self.v_srv['token']))
    self.v_rdy = json.loads(await self.ws.recv())
    self.ssrc = self.v_rdy['d']['ssrc']

In [ ]:
await vc._connect(); vc

**IP Discovery**: Discord needs your *external* IP/port (what the internet sees after NAT), not your local one. We send a 74-byte UDP request containing our SSRC, and Discord responds with our external address filled in:

| Field | Size | Description |
|-------|------|-------------|
| Type | 2 bytes | 1=request, 2=response |
| Length | 2 bytes | 70 (size of remaining fields) |
| SSRC | 4 bytes | Our audio stream identifier |
| Address | 64 bytes | Blank in request; our external IP in response |
| Port | 2 bytes | Blank in request; our external port in response |

Note: Discord requires a **Speaking** event (even with `speaking=0`) before it will send audio packets to you.

In [ ]:
#| export
async def get_ip(trans, proto, ssrc):
    pkt = bytearray(74)
    pkt[0:2],pkt[2:4],pkt[4:8] = (1).to_bytes(2,'big'),(70).to_bytes(2,'big'),ssrc.to_bytes(4,'big')
    trans.sendto(pkt)
    r = await proto.packets.get()
    return r[8:72].decode('utf-8').rstrip('\x00'), int.from_bytes(r[72:74], 'big')

`_udp` sets up the audio transport: open UDP socket, discover our external IP, tell Discord which encryption mode to use (`select_protocol`), receive the secret key, and signal that we're ready to speak.

In [ ]:
#| export
class VoiceUDP(asyncio.DatagramProtocol):
    def __init__(self): self.packets = asyncio.Queue()
    def datagram_received(self, data, addr): self.packets.put_nowait(data)

In [ ]:
#| export
@patch
async def _udp(self:VoiceClient):
    udp_ip, udp_port = self.v_rdy['d']['ip'], self.v_rdy['d']['port']
    self.loop = asyncio.get_running_loop()
    self.trans, self.proto = await self.loop.create_datagram_endpoint(VoiceUDP, remote_addr=(udp_ip, udp_port))
    ip, port = await get_ip(self.trans, self.proto, self.ssrc)
    await self.ws.send(Op.select_protocol(ip, port))
    desc = json.loads(await self.ws.recv())
    while desc['op'] != 4: desc = json.loads(await self.ws.recv())
    self.secret = bytes(desc['d']['secret_key'])
    await self.ws.send(Op.speaking(self.ssrc))

In [ ]:
await vc._udp()

In [ ]:
# vc.secret

`join` ties the three phases together into a single call: connect to voice server, set up UDP, ready to receive audio.

In [ ]:
#| export
@patch
async def join(self:VoiceClient):
    await self._connect()
    await self._udp()
    print(f"Voice ready!")

In [ ]:
await vc.join()

Voice ready!


Audio arrives as UDP packets using the **RTP (Real-time Transport Protocol)** format. Each packet contains:
- **RTP header** (12+ bytes) — version, sequence number, timestamp, SSRC (identifies which user is speaking)
- **Encrypted payload** — the Opus-encoded audio, encrypted with the `secret_key`
- **Nonce** (4 bytes at end) — used for decryption

Decryption with `aead_xchacha20_poly1305_rtpsize`:
- The cipher expects a 24-byte nonce, but only 4 bytes are transmitted — we pad with 20 zero bytes
- The unencrypted RTP header is used as AAD (Additional Authenticated Data) — it's verified but not encrypted
- For `rtpsize` mode, the AAD includes the 12-byte base header, any CSRC entries (4 bytes each, usually 0), and only the 4-byte extension *preamble* — the extension *data* is part of the encrypted payload
- After decryption, we skip past the extension data (its length is in bytes 14-15 of the original packet, multiplied by 4) to reach the raw Opus audio

Packets with byte[1] == `0x78` are RTP voice data. Other packets (like `0xC9`) are RTCP control packets used for connection quality reporting — we skip those.

In [ ]:
#| export
def decrypt_pkt(pkt, secret):
    csrc_cnt = pkt[0] & 0x0F
    hdr_sz = 12 + csrc_cnt * 4 + (4 if pkt[0] & 0x10 else 0)
    nonce = bytearray(24) # only use 4, but pad to 24 bytes as expected by the cipher
    nonce[:4] = pkt[-4:]
    data, hdr, nonce = bytes(pkt[hdr_sz:-4]), bytes(pkt[:hdr_sz]), bytes(nonce)
    decrypted = xchacha_decrypt(data, hdr, nonce, bytes(secret))
    ext_sz = int.from_bytes(pkt[14:16], 'big') * 4 # extension data size in bytes
    return decrypted[ext_sz:]

Recording writes each speaker to a separate file (`recording_<ssrc>.mp3`) rather than mixing in real-time. This keeps the code simple and gives us per-speaker files — ideal for transcription. Each speaker gets their own Opus decoder because Opus is *stateful* — feeding multiple speakers through one decoder corrupts its internal state and produces garbled audio.

Silence padding ensures all files stay time-aligned: at the start (padding from `_rec_start` to first packet), and during gaps (using RTP timestamps, which Discord increments continuously even during silence). Without this, files would be compressed in time and impossible to merge or correlate.

In [ ]:
#| export
spf = 960
sr = 48_000
def silence(n_smpls:int): return b'\x00' * (n_smpls * 2) # 2 bytes per sample (s16le)

@patch
def _get_proc(self:VoiceClient, ssrc):
    if ssrc not in self._procs:
        pth = str(self._rec_path.with_stem(f"{self._rec_path.stem}_{ssrc}"))
        proc = (ffmpeg.input('pipe:', f='s16le', ar=sr, ac=1)
                      .output(pth).overwrite_output()
                      .run_async(pipe_stdin=True, quiet=True))
        proc.stdin.write(silence(int((time.time() - self._rec_start) * sr)))
        self._procs[ssrc] = proc
    return self._procs[ssrc]

@patch
def _write_audio(self:VoiceClient, ssrc, ts, data):
    proc = self._get_proc(ssrc)
    # Fill silence gaps
    if ssrc in self._last_ts:
        gap = ts - self._last_ts[ssrc] - spf
        if gap > 0: proc.stdin.write(silence(gap))
    self._last_ts[ssrc] = ts
    proc.stdin.write(self._decoders[ssrc].decode(data, spf))

@patch
async def _recv_audio(self:VoiceClient):
    while self._recording:
        try:
            pkt = await asyncio.wait_for(self.proto.packets.get(), timeout=0.1)
            if pkt[1] != 0x78: continue # filter out RTP non-voice packets 
            ssrc = int.from_bytes(pkt[8:12], 'big')
            ts = int.from_bytes(pkt[4:8], 'big')
            if ssrc not in self._decoders: self._decoders[ssrc] = opuslib_next.Decoder(sr, 1)
            if data := decrypt_pkt(pkt, self.secret): self._write_audio(ssrc, ts, data)
        except asyncio.TimeoutError: continue

`start_recording` drains any previously queued packets so recordings start clean. `stop_recording` calls `communicate()` on each ffmpeg process to flush buffered audio and cleanly close the output files — without this, recordings may be truncated or corrupted.

In [ ]:
#| export
@patch
def start_recording(self:VoiceClient, path='recording.mp3'):
    while not self.proto.packets.empty(): self.proto.packets.get_nowait()
    self._rec_path,self._decoders,self._procs,self._last_ts = Path(path),{},{},{}
    self._rec_start = time.time()
    self._recording = True
    self._rec_task = asyncio.create_task(self._recv_audio())
    return path

@patch
def stop_recording(self:VoiceClient):
    self._recording = False
    self._rec_task.cancel()
    for p in self._procs.values(): p.communicate()
    return [str(self._rec_path.with_stem(f"{self._rec_path.stem}_{ssrc}")) for ssrc in self._procs]

In [ ]:
pth = '/tmp/recording.mp3'
vc.start_recording(path=pth)
await asyncio.sleep(10)
rpths = vc.stop_recording(); rpths

In [ ]:
# Audio(rpths[0])

To leave a voice channel cleanly:
1. Stop the heartbeat loop
2. Close the Voice WebSocket
3. Close the UDP transport
4. Send a Voice State Update with `channel_id=None` on the main gateway — this tells Discord to remove the bot from the voice channel. Without this, the bot appears to stay in the channel until it times out.

In [ ]:
#| export
@patch
async def leave(self:VoiceClient):
    self.running = False
    if self.ws: await self.ws.close()
    if self.trans: self.trans.close()
    await self.gw.ws.send(Op.voice_state(self.ch.guild_id, None))

In [ ]:
await vc.leave()

In [ ]:
await gc.stop()

Gateway stopped!


## Bots

`Bot` ties together `DiscordClient` (REST) and `GatewayClient` (events) into a single object with a decorator-based command router. Commands are registered with `@bot.cmd` — the function name becomes the command name, prefixed with `!` in Discord. So `def echo` responds to `!echo`. Every command handler takes two arguments: the `Message` object and a string of everything the user typed after the command name (empty string if nothing).

The `_on_msg` handler ignores messages from the bot itself to prevent infinite loops — a common gotcha with Discord bots. It splits the message into command name + args, so `!echo hello world` passes `"hello world"` as the `args` string to the handler.

In [ ]:
#| export
class Bot:
    "Discord bot with command routing"
    def __init__(self, intents, **kw):
        self.dc = DiscordClient(**kw)
        self.gc = GatewayClient(intents, self.dc)
        self.cmds = {}
        self.errors = []
        self._on_err = None

    def cmd(self, f):
        "Register a command handler — function name becomes !name"
        self.cmds[f.__name__] = f
        return f

    async def _on_msg(self, msg):
        if msg.author['id'] == self.gc.user_id: return
        parts = msg.content.split(maxsplit=1)
        if not parts or not parts[0].startswith('!'): return
        name = parts[0][1:]
        if name not in self.cmds: return
        try:
            res = self.cmds[name](msg, parts[1] if len(parts) > 1 else '')
            if asyncio.iscoroutine(res): await res
        except Exception as e:
            self.errors.append(e)
            if self._on_err: await self._on_err(msg, e)

    async def start(self):
        "Connect to gateway and start listening"
        await self.gc.start()
        self.gc.on('MESSAGE_CREATE', self._on_msg)

    async def stop(self): await self.gc.stop()
    def __repr__(self): return f"Bot(cmds={list(self.cmds.keys())})"

In [ ]:
bot = Bot(intents)
await bot.start()

Connected! Session: aebd851f6f7e226c5f982f03cf7d27ef, heartbeat: 41250ms
Gateway started!


Commands can be registered at any time — even after `bot.start()`. This works because `@bot.cmd` just adds the function to a dict; the message handler looks up commands dynamically on each message.

In [ ]:
@bot.cmd
async def echo(msg, args): await (await msg.channel()).send(f'You said: {args}')

In [ ]:
bot

Bot(cmds=['echo'])

Errors in command handlers are caught and stored in `bot.errors` — useful for debugging in dynamic environments like solveit where you can inspect the list after the fact. For real-time handling (e.g. notifying the user in Discord), register a handler with `@bot.on_error`. Both mechanisms work simultaneously.

In [ ]:
#| export
@patch
def on_error(self:Bot, f):
    self._on_err = f
    return f

In [ ]:
@bot.on_error
async def handle_err(msg, e): print('error')

In [ ]:
@bot.cmd
def err(msg): raise Exception('test')

In [ ]:
bot.errors

[]

Voice integration reuses the existing `VoiceClient` — `Bot` just provides convenience methods to manage the lifecycle. The bot can only be in one voice channel at a time.

In [ ]:
#| export
@patch
async def join_voice(self:Bot, channel):
    "Join a voice channel and return VoiceClient"
    self.vc = VoiceClient(self.gc, channel)
    await self.vc.join()
    return self.vc

@patch
async def leave_voice(self:Bot):
    "Leave the current voice channel"
    if self.vc: await self.vc.leave()
    self.vc = None

In [ ]:
vc = await bot.join_voice(vch); vc

Voice ready!


VoiceClient(self.ch=Channel(id=1327046393453613080, name='General', type=2), self.running=True)

In [ ]:
vc.start_recording()

'recording.mp3'

In [ ]:
pth = vc.stop_recording()
await bot.leave_voice()

In [ ]:
# Audio(pth)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()